# 2110446 DATA SCIENCE AND DATA ENGINEERING

## **Unit 07:** Data Ingestion

- **Author:** Worralop Srichainont
- **Year:** 2025 (Semester 2)

## **Homework:** Data Ingestion

Each student will receive verifier id (vid) and token from TA. (Discord)

Modify the Transaction Verifier notebook in the git repo (link) to connect to the Kafka broker at IP address = lab.aimet.tech (port = 9092) and performs the following task

1. Read a transaction from “transaction” topic using transaction.avsc schema, extract transaction information including txid, payer, payee, and amount
2. Verify transaction information with your token by creating signature using gen_signature function (given in the assignment) and submit your verification (vid, txid, and signature) to “submit” topic using submit.avsc schema
3. System receives and verify the signature and send the confirmation back to verifier via “result” topic
4. Verifier receives a confirmation from “result” topic (message with code “200” with matching vid and txid)

# Dependencies

In [1]:
%pip install avro kafka-python

In [2]:
import gdown
import avro.schema
import avro.io
import io
import hashlib, json

from kafka import KafkaConsumer, KafkaProducer

# Constants

In [ ]:
VID = "V970892"
TOKEN = "0ccf12b56e767b1a4d71cbc72a9ab98d"

KAFKA_BROKER = "lab.aimet.tech:9092"

TX_SCHEMA_FILE_URL = "https://drive.google.com/uc?id=1AUOBNEjlpE3UInxDKA_8kjGIwnGsNwM1"
SUBMIT_SCHEMA_FILE_URL = (
    "https://drive.google.com/uc?id=1Ie9FsdOFdAdBg5aZO4Vsc4T_9lCLYHGG"
)
RESULT_SCHEMA_FILE_URL = (
    "https://drive.google.com/uc?id=1GxpCs2NPxYHrYpbn5xahlIrKFCdIjvT4"
)

TX_SCHEMA_FILE_PATH = "transaction.avsc"
SUBMIT_SCHEMA_FILE_PATH = "submit.avsc"
RESULT_SCHEMA_FILE_PATH = "result.avsc"

# Functions

In [4]:
def serialize(schema, obj):
    bytes_writer = io.BytesIO()
    encoder = avro.io.BinaryEncoder(bytes_writer)
    writer = avro.io.DatumWriter(schema)
    writer.write(obj, encoder)
    return bytes_writer.getvalue()

In [5]:
def deserialize(schema, raw_bytes):
    bytes_reader = io.BytesIO(raw_bytes)
    decoder = avro.io.BinaryDecoder(bytes_reader)
    reader = avro.io.DatumReader(schema)
    return reader.read(decoder)

In [6]:
def gen_signature(txid, payer, payee, amount, token):
    o = {"txid": txid, "payer": payer, "payee": payee, "amount": amount, "token": token}
    return hashlib.md5(json.dumps(o, sort_keys=True).encode("utf-8")).hexdigest()

In [ ]:
def display_transaction_msg(transaction_msg):
    tx_data = transaction_msg.value
    print(f"- txid:   {tx_data['txid']}")
    print(f"- payer:  {tx_data['payer']}")
    print(f"- payee:  {tx_data['payee']}")
    print(f"- amount: {tx_data['amount']}")

In [8]:
def get_submit_payload(transaction_msg):
    # Get values
    tx_data = transaction_msg.value
    txid = tx_data["txid"]
    payer = tx_data["payer"]
    payee = tx_data["payee"]
    amount = tx_data["amount"]

    # Create signature
    signature = gen_signature(txid, payer, payee, amount, TOKEN)

    # Create submit message
    submit_msg = {"vid": VID, "txid": txid, "signature": signature}

    # Create submit payload
    serialized_submit = serialize(submitschema, submit_msg)
    return serialized_submit

In [9]:
def display_verification_msg(reult_msg):
    result_data = reult_msg.value
    print(f"- timestamp: {result_data["timestamp"]}")
    print(f"- vid:       {result_data["vid"]}")
    print(f"- txid:      {result_data["txid"]}")
    print(f"- checksum:  {result_data["checksum"]}")
    print(f"- code:      {result_data["code"]}")
    print(f"- message:   {result_data["message"]}")

# Load Files

In [10]:
gdown.download(TX_SCHEMA_FILE_URL, TX_SCHEMA_FILE_PATH, quiet=False)
gdown.download(SUBMIT_SCHEMA_FILE_URL, SUBMIT_SCHEMA_FILE_PATH, quiet=False)
gdown.download(RESULT_SCHEMA_FILE_URL, RESULT_SCHEMA_FILE_PATH, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1AUOBNEjlpE3UInxDKA_8kjGIwnGsNwM1
To: /content/transaction.avsc
100%|██████████| 285/285 [00:00<00:00, 853kB/s]
Downloading...
From: https://drive.google.com/uc?id=1Ie9FsdOFdAdBg5aZO4Vsc4T_9lCLYHGG
To: /content/submit.avsc
100%|██████████| 240/240 [00:00<00:00, 907kB/s]
Downloading...
From: https://drive.google.com/uc?id=1GxpCs2NPxYHrYpbn5xahlIrKFCdIjvT4
To: /content/result.avsc
100%|██████████| 371/371 [00:00<00:00, 1.39MB/s]


'result.avsc'

In [11]:
txschema = avro.schema.parse(open(TX_SCHEMA_FILE_PATH).read())
submitschema = avro.schema.parse(open(SUBMIT_SCHEMA_FILE_PATH).read())
resultschema = avro.schema.parse(open(RESULT_SCHEMA_FILE_PATH).read())

# Initialize Components

In [12]:
producer = KafkaProducer(bootstrap_servers=[KAFKA_BROKER])

In [13]:
txconsumer = KafkaConsumer(
    "transaction",
    bootstrap_servers=[KAFKA_BROKER],
    enable_auto_commit=True,
    value_deserializer=lambda x: deserialize(txschema, x),
)

In [14]:
resultconsumer = KafkaConsumer(
    "result",
    bootstrap_servers=[KAFKA_BROKER],
    enable_auto_commit=True,
    value_deserializer=lambda x: deserialize(resultschema, x),
)

# Implementation

In [15]:
for tx_message in txconsumer:
    # Display transaction message
    print("TRANSACTION")
    display_transaction_msg(tx_message)

    # Get transaction ID
    txid = tx_message.value["txid"]

    # Submit to submit topic
    submit_payload = get_submit_payload(tx_message)
    producer.send("submit", submit_payload)
    producer.flush()

    # Display verification message
    for res_message in resultconsumer:
        res_data = res_message.value

        if res_data["vid"] == VID and res_data["txid"] == txid:
            if res_data["code"] == 200:
                print("VERIFICATION SUCCESS")
            else:
                print("VERIFICATION FAILED")

            display_verification_msg(res_message)
            break
    break

TRANSACTION
- txid:   TX02618
- payer:  A79924
- payee:  A59221
- amount: 1140
VERIFICATION SUCCESS
- timestamp: 1775015394
- vid:       V970892
- txid:      TX02618
- checksum:  d15ef9616b704ffc17f6c3ea8f8e6f89
- code:      200
- message:   Confirm
